## First Project: The Digital Twin

- Pushover is a nifty tool for sending Push Notification to your phone
- It's super easy to set up and install !!!



In [1]:
from dotenv import load_dotenv
from openai import OpenAI
import json
import os
import requests
from pypdf import PdfReader
import gradio as gr

d:\2026-courses\agent-track\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
load_dotenv(override=True)

True

In [3]:
OLLAMA_BASE_URL = os.getenv("OLLAMA_BASE_URL")


In [4]:
ollama = OpenAI(base_url=OLLAMA_BASE_URL, api_key="ollama")

In [5]:
pushover_user = os.getenv("PUSHOVER_USER")
pushover_token = os.getenv("PUSHOVER_TOKEN")
pushover_url = os.getenv("PUSHOVER_URL")

In [6]:
if pushover_user:
    if pushover_user.startswith("u"):
        print("Pushover user found and looks good")
    
    else:
        print("Pushover user found but doesn't start with u")

else :
    print("No Pushover user found")


if pushover_token:
    if pushover_token.startswith("a"):
        print("Pushover token found and looks good")
    else:
        print("Pushover token found but doesn't start with a")
else:
    print("Pushover token not found")



Pushover user found and looks good
Pushover token found and looks good


In [7]:
def push(message):
    print(f"Push: {message}")
    payload = {"user": pushover_user, "token": pushover_token, "message": message}
    requests.post(pushover_url, data=payload)


In [8]:
push("HEY!!")

Push: HEY!!


In [9]:
def record_user_details(email, name="Name not provided", notes="Not provided"):
    push(f"Recording interest from {name} with email {email} and notes {notes}")
    return "OK"


In [10]:
def record_unkown_question(question):
    push(f"Recording {question} asked that I couldn't answer")
    return "OK"

In [ ]:
record_user_details_json={
    "name": "record_user_details",
    "description": "Use this tool to record that a user is interested in being in touch and provided an email address",
    "parameters":{
        "type": "object",
        "properties":{
            "email": {"type": "string", "description": "The email address of the user"},
            "name": {"type": "string", "description": "The name of the user"},
            "notes": {"type": "string", "description": "Any additional notes about the user"}
        },
        "required": ["email"],
        "additionalProperties": False
    }
}

In [12]:
record_unkown_question_json={
    "name": "record_unkown_question",
    "description": "Always use this tool to record any question that couldn't be answered as you didn't know the answer",
    "parameters":{
        "type": "object",
        "properties":{
            "question": {"type": "string", "description": "The question that couldn't be answered"}
        },
        "required": ["question"],
        "additionalProperties": False
    }
}

In [13]:
tools = [
    {"type": "function", "function": record_user_details_json},
    {"type": "function", "function": record_unkown_question_json}
]


tools



[{'type': 'function',
  'function': {'name': 'record_user_details',
   'description': 'Use this tool to record that a user is interested in being in touch and provided an email address',
   'parameters': {'type': 'object',
    'properties': {'email': {'type': 'string',
      'description': 'The email address of the user'},
     'name': {'type': 'string', 'description': 'The name of the user'},
     'notes': {'type': 'string',
      'description': 'Any additional notes about the user'}},
    'required': ['email'],
    'additionalProperties': False}}},
 {'type': 'function',
  'function': {'name': 'record_unkown_question',
   'description': "Always use this tool to record any question that couldn't be answered as you didn't know the answer",
   'parameters': {'type': 'object',
    'properties': {'question': {'type': 'string',
      'description': "The question that couldn't be answered"}},
    'required': ['question'],
    'additionalProperties': False}}}]

In [44]:
# This function can take a list of tool calls, and run them. This is the IF statement!!

def handle_tool_calls_with_manual_if(tool_calls):
    results = []
    for tool_call in tool_calls:
        print(tool_call["function"]["name"])
        tool_name =tool_call["function"]["name"]
        arguments = json.loads(tool_call["function"]["parameters"]['properties'])
        print(f"Tool called: {tool_name}", flush=True)

        # THE BIG IF STATEMENT!!!

        if tool_name == "record_user_details":
            result = record_user_details(**arguments)
        elif tool_name == "record_unknown_question":
            result = record_unknown_question(**arguments)

        results.append({"role": "tool","content": json.dumps(result),"tool_call_id": tool_call.id})
    return results


In [ ]:
handle_tool_calls_with_manual_if(tools)

## Using Python built-in globals()

Python has a dictionary that gives us access to all global functions

Sidenote: for sure when we deploy, we will use this in more predicted way...

In [46]:
globals()["record_unknown_question"]("this is a really hard question")

KeyError: 'record_unknown_question'

In [20]:
def handle_tool_calls(tool_calls):
    results = []
    for tool_call in tool_calls:
        tool_name = tool_call.function.name
        arguments = json.loads(tool_call.function.arguments)
        print(f"Tool called: {tool_name}", flush=True)
        tool = globals().get(tool_name)
        result = tool(**arguments) if tool else "No tool found"
        results.append({"role": "tool","content": json.dumps(result),"tool_call_id": tool_call.id})
    return results

In [21]:
reader = PdfReader("../../my_profile/Profile.pdf")
linkedin = ""
for page in reader.pages:
    text = page.extract_text()
    if text:
        linkedin += text

with open("../../my_profile/summary.txt", "r", encoding="utf-8") as f:
    summary = f.read()

In [25]:
system_prompt = f"""

# Your role

You are a digital twin running on a website, chatting with visitors of the website.
You represent the person who's website you are on.
You answer questions related to their career, background, skills and experience.

Here are the details of the person you are representing:

{summary}

If asked, you explain clearly that you are an AI that is the digital twin of this person.

# Context

Here is a summary of the person's LinkedIn profile so that you can answer questions:

{linkedin}

# Rules

Engage with the user. Be professional and engaging, as if talking to a potential client or future employer who came across the website.
Only answer questions related to career, background, skills and experience.
If the user asks about something unrelated, then steer the conversation back to professional topics.

Always stay in character as the digital twin of the person you are representing. Represent the person.

If the user would like to get in touch, then ask for their email, and use your tool to record their email for follow-up.

If you don't know the answer for the question asked by the user use the tool called record_unkown_question.

IMPORTANT:
If you don't know the answer, use your tool to record the question, and then tell the user that you don't know. Never make up an answer.
"""

In [27]:
def chat(message, history):
    messages = [{"role": "system", "content": system_prompt}] + history + [{"role": "user", "content": message}]

    response = ollama.chat.completions.create(
        model= "qwen2.5:7b",
        messages= messages,
        tools= tools,
    )

    while response.choices[0].finish_reason == "tool_calls":
        message = response.choices[0].message
        tool_calls = message.tool_calls
        results = handle_tool_calls(tool_calls)
        messages.append(message)
        messages.extend(results)

        response = ollama.chat.completions.create(
        model= "qwen2.5:7b",
        messages= messages,
        tools= tools,
    )

    return response.choices[0].message.content







In [28]:
gr.ChatInterface(chat).launch(inbrowser=True)

* Running on local URL:  http://127.0.0.1:7862
* To create a public link, set `share=True` in `launch()`.


Tool called: record_unkown_question
Push: Recording What are the stock prices of HUDHUD AI? asked that I couldn't answer
Tool called: record_unkown_question
Push: Recording What are the questions that you can't answer? asked that I couldn't answer
Tool called: record_unkown_question
Push: Recording What are the questions that you can't answer? asked that I couldn't answer
